In [2]:
   df = pd.read_csv("diabetic_data.csv", na_values="?", low_memory=False)

In [3]:
import time
import pandas as pd

In [6]:
# ----------------------------------------------------------------------------
# 0. LOAD THE DATA
# ----------------------------------------------------------------------------
# In this file, missing values are written as "?", so na_values="?" tells pandas
# to treat them as proper missing values (NaN).
# low_memory=False avoids a warning about columns with mixed data types.
df = pd.read_csv("diabetic_data.csv", na_values="?", low_memory=False)

print("Rows, columns:", df.shape)
print(df[["encounter_id", "age", "gender", "time_in_hospital",
          "num_medications", "insulin", "readmitted"]].head())

Rows, columns: (101766, 50)
   encounter_id      age  gender  time_in_hospital  num_medications insulin  \
0       2278392   [0-10)  Female                 1                1      No   
1        149190  [10-20)  Female                 3               18      Up   
2         64410  [20-30)  Female                 2               13      No   
3        500364  [30-40)    Male                 2               16      Up   
4         16680  [40-50)    Male                 1                8  Steady   

  readmitted  
0         NO  
1        >30  
2         NO  
3         NO  
4         NO  


In [7]:
# ----------------------------------------------------------------------------
# 1. MELT  (wide -> long)
# ----------------------------------------------------------------------------
# The data has one column per diabetes medication (metformin, insulin, ...).
# That is a "wide" layout. melt() stacks those columns into two columns:
#   - "medication"      (the old column name)
#   - "dosage_status"   (the old cell value: No, Steady, Up, Down)
medication_columns = ["metformin", "glipizide", "glyburide", "pioglitazone", "insulin"]

melted = df.melt(
    id_vars=["encounter_id", "age", "readmitted"],  # columns to keep as they are
    value_vars=medication_columns,                  # columns to stack
    var_name="medication",
    value_name="dosage_status",
)

In [8]:
print("\n--- MELT ---")
print("Before melt:", df.shape, "-> After melt:", melted.shape)
print(melted.head())

# Use the long format: how many hospital visits involved each medication?
# (each row is one hospital visit; a status of "No" means not on that medication)
prescribed = melted[melted["dosage_status"] != "No"]
print("\nHospital visits involving each medication:")
print(prescribed["medication"].value_counts())


--- MELT ---
Before melt: (101766, 50) -> After melt: (508830, 5)
   encounter_id      age readmitted medication dosage_status
0       2278392   [0-10)         NO  metformin            No
1        149190  [10-20)        >30  metformin            No
2         64410  [20-30)         NO  metformin            No
3        500364  [30-40)         NO  metformin            No
4         16680  [40-50)         NO  metformin            No

Hospital visits involving each medication:
medication
insulin         54383
metformin       19988
glipizide       12686
glyburide       10650
pioglitazone     7328
Name: count, dtype: int64


In [9]:
# ----------------------------------------------------------------------------
# 2. PIVOT  (long -> wide)
# ----------------------------------------------------------------------------
# pivot() reshapes a table so values become columns. It needs each
# (index, column) pair to appear only ONCE, so first we make a summary table
# with groupby: one row per (age group, readmission status).
avg_stay = (
    df.groupby(["age", "readmitted"])["time_in_hospital"]
    .mean()
    .reset_index()          # turn the group labels back into normal columns
)

In [10]:
print("\n--- PIVOT ---")
print("Long table (before pivot):")
print(avg_stay.head(6))


--- PIVOT ---
Long table (before pivot):
       age readmitted  time_in_hospital
0   [0-10)        <30          2.666667
1   [0-10)        >30          2.923077
2   [0-10)         NO          2.469697
3  [10-20)        <30          3.625000
4  [10-20)        >30          3.767857
5  [10-20)         NO          2.847775


In [11]:
# Age groups go down the side, readmission status goes across the top.
pivoted = avg_stay.pivot(
    index="age",
    columns="readmitted",
    values="time_in_hospital",
)
print("\nAverage days in hospital by age group and readmission status:")
print(pivoted.round(2))


Average days in hospital by age group and readmission status:
readmitted   <30   >30    NO
age                         
[0-10)      2.67  2.92  2.47
[10-20)     3.62  3.77  2.85
[20-30)     3.50  3.84  3.43
[30-40)     4.16  4.12  3.55
[40-50)     4.55  4.36  3.75
[50-60)     4.71  4.38  3.87
[60-70)     4.71  4.51  4.23
[70-80)     4.99  4.56  4.52
[80-90)     4.97  4.72  4.83
[90-100)    4.76  4.51  4.87


In [12]:
# Bonus: pivot_table() does the averaging AND the pivot in a single step,
# so it works even when (index, column) pairs repeat.
pivot_table_version = df.pivot_table(
    index="gender",
    columns="readmitted",
    values="num_medications",
    aggfunc="mean",
)
print("\npivot_table: average number of medications by gender and readmission:")
print(pivot_table_version.round(2))


pivot_table: average number of medications by gender and readmission:
readmitted         <30    >30     NO
gender                              
Female           17.14  16.53  15.75
Male             16.62  15.98  15.58
Unknown/Invalid    NaN    NaN  16.33


In [13]:
# ----------------------------------------------------------------------------
# 3. AGGREGATION  (summarize many rows into a few numbers)
# ----------------------------------------------------------------------------
# .agg() lets us ask for several statistics at once.
numeric_columns = ["time_in_hospital", "num_medications", "num_lab_procedures"]

summary_stats = df[numeric_columns].agg(["count", "mean", "median", "min", "max", "std"])

print("\n--- AGGREGATION ---")
print("Several statistics for several columns:")
print(summary_stats.round(2))

# A dictionary lets us choose different statistics for each column.
custom_agg = df.agg({
    "time_in_hospital": ["mean", "max"],
    "num_medications": ["median", "min", "max"],
    "number_inpatient": ["sum"],
})
print("\nDifferent statistics per column (dictionary):")
print(custom_agg)


--- AGGREGATION ---
Several statistics for several columns:
        time_in_hospital  num_medications  num_lab_procedures
count          101766.00        101766.00           101766.00
mean                4.40            16.02               43.10
median              4.00            15.00               44.00
min                 1.00             1.00                1.00
max                14.00            81.00              132.00
std                 2.99             8.13               19.67

Different statistics per column (dictionary):
        time_in_hospital  num_medications  number_inpatient
mean            4.395987              NaN               NaN
max            14.000000             81.0               NaN
median               NaN             15.0               NaN
min                  NaN              1.0               NaN
sum                  NaN              NaN           64679.0


In [14]:
# ----------------------------------------------------------------------------
# 4. ITERATION  (looping through the data)
# ----------------------------------------------------------------------------
# Use a small slice of rows so the output stays readable.
small = df[["encounter_id", "age", "time_in_hospital", "num_medications"]].head(5)

print("\n--- ITERATION ---")

# (a) iterrows(): gives (row_number, row) pairs. Flexible but slow.
print("iterrows:")
for row_number, row in small.iterrows():
    print(f"  Row {row_number}: encounter {row['encounter_id']}, "
          f"stayed {row['time_in_hospital']} days")

# (b) itertuples(): gives each row as a named tuple. Faster than iterrows.
print("itertuples:")
for row in small.itertuples():
    print(f"  Age group {row.age} took {row.num_medications} medications")

# (c) items(): loops over COLUMNS instead of rows.
print("items (columns):")
for column_name, column_values in small.items():
    print(f"  {column_name}: first value is {column_values.iloc[0]}")

# Loops are useful, but pandas "vectorized" code is much faster. Here we label
# each stay as "long" (more than 7 days) both ways, and time them.
sample = df.head(20000)

start = time.perf_counter()
labels_loop = []
for row in sample.itertuples():
    labels_loop.append("long" if row.time_in_hospital > 7 else "short")
loop_seconds = time.perf_counter() - start

start = time.perf_counter()
labels_vectorized = (sample["time_in_hospital"] > 7).map({True: "long", False: "short"})
vector_seconds = time.perf_counter() - start

print(f"\nLabelling 20,000 stays with a loop:       {loop_seconds:.4f} seconds")
print(f"Labelling 20,000 stays vectorized:        {vector_seconds:.4f} seconds")
print("Same result either way:", labels_loop == labels_vectorized.tolist())


--- ITERATION ---
iterrows:
  Row 0: encounter 2278392, stayed 1 days
  Row 1: encounter 149190, stayed 3 days
  Row 2: encounter 64410, stayed 2 days
  Row 3: encounter 500364, stayed 2 days
  Row 4: encounter 16680, stayed 1 days
itertuples:
  Age group [0-10) took 1 medications
  Age group [10-20) took 18 medications
  Age group [20-30) took 13 medications
  Age group [30-40) took 16 medications
  Age group [40-50) took 8 medications
items (columns):
  encounter_id: first value is 2278392
  age: first value is [0-10)
  time_in_hospital: first value is 1
  num_medications: first value is 1

Labelling 20,000 stays with a loop:       0.2086 seconds
Labelling 20,000 stays vectorized:        0.0024 seconds
Same result either way: True


In [15]:
# ----------------------------------------------------------------------------
# 5. GROUPBY  (split into groups, apply a calculation, combine the results)
# ----------------------------------------------------------------------------
print("\n--- GROUPBY ---")

# (a) One grouping column, one statistic.
print("Average days in hospital by readmission status:")
print(df.groupby("readmitted")["time_in_hospital"].mean().round(2))

# (b) Named aggregation: several statistics, each with a clear column name.
by_readmission = df.groupby("readmitted").agg(
    visits=("encounter_id", "count"),
    avg_days=("time_in_hospital", "mean"),
    avg_medications=("num_medications", "mean"),
    max_inpatient_visits=("number_inpatient", "max"),
)
print("\nSummary by readmission status:")
print(by_readmission.round(2))

# (c) Two grouping columns.
by_gender_readmit = df.groupby(["gender", "readmitted"])["time_in_hospital"].mean()
print("\nAverage days by gender and readmission status:")
print(by_gender_readmit.round(2))

# (d) transform(): returns a value for EVERY row, so we can add it as a column.
# Here: how does each visit's stay compare to the average for their age group?
df["age_group_avg_stay"] = df.groupby("age")["time_in_hospital"].transform("mean")
df["stay_vs_age_avg"] = df["time_in_hospital"] - df["age_group_avg_stay"]
print("\nStay compared with the age-group average (first 5 rows):")
print(df[["age", "time_in_hospital", "age_group_avg_stay", "stay_vs_age_avg"]].head().round(2))

# (e) Looping over groups: each iteration gives the group name and its rows.
print("\nLooping over groups (visits per readmission status):")
for status, group in df.groupby("readmitted"):
    print(f"  {status}: {len(group)} visits")


--- GROUPBY ---
Average days in hospital by readmission status:
readmitted
<30    4.77
>30    4.50
NO     4.25
Name: time_in_hospital, dtype: float64

Summary by readmission status:
            visits  avg_days  avg_medications  max_inpatient_visits
readmitted                                                         
<30          11357      4.77            16.90                    21
>30          35545      4.50            16.28                    19
NO           54864      4.25            15.67                    16

Average days by gender and readmission status:
gender           readmitted
Female           <30           4.83
                 >30           4.60
                 NO            4.33
Male             <30           4.70
                 >30           4.37
                 NO            4.17
Unknown/Invalid  NO            3.33
Name: time_in_hospital, dtype: float64

Stay compared with the age-group average (first 5 rows):
       age  time_in_hospital  age_group_avg_stay  st